<a href="https://colab.research.google.com/github/sonky20/sonky/blob/master/3%EC%9D%BC%EC%B0%A8_%ED%9A%8C%EA%B7%80%EB%AA%A8%EB%8D%B8%EC%8B%A4%EC%8A%B51.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import *
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")


Using cpu device


In [8]:
#data loading
path = 'https://bit.ly/ds_boston_csv'
data = pd.read_csv(path)
#data.head()

#seperate x, y
target = 'medv'
features = ['lstat', 'ptratio', 'crim']
x = data.loc[:, features]
y = data.loc[:, target]

#seperate  train, val
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=.2, random_state=20)

#declare scaler
scaler = MinMaxScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)

In [9]:
def make_DataSet(x_train, y_train, x_val, y_val, batch_size=32):
  #translate to data tensor
  x_train_tensor = torch.tensor(x_train, dtype=torch.float32)
  y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
  x_val_tensor = torch.tensor(x_val, dtype=torch.float32)
  y_val_tensor = torch.tensor(y_val, dtype=torch.float32).view(-1, 1)

  #Create TensorDataset: Combine to tensor data set
  train_dataset = TensorDataset(x_train_tensor, y_train_tensor)

  #Create DataLoader
  train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
  return train_loader, x_val_tensor, y_val_tensor

In [10]:
print(type(y_train), type(y_val))
y_train = y_train.values
y_val = y_val.values
print(type(y_train), type(y_val))

<class 'pandas.core.series.Series'> <class 'pandas.core.series.Series'>
<class 'numpy.ndarray'> <class 'numpy.ndarray'>


In [11]:
train_loader, x_val_ts, y_val_ts = make_DataSet(x_train, y_train, x_val, y_val)

#첫번째 배치만 로딩해서 살펴보기
for x, y in train_loader:
  print(f"Shape of x [rows, columns]: {x.shape}")
  print(f"Shape of y: {y.shape} {y.dtype}")
  break

Shape of x [rows, columns]: torch.Size([32, 3])
Shape of y: torch.Size([32, 1]) torch.float32


In [12]:
#print(x.shape[1])
n_feature = x.shape[1]
model1 = nn.Sequential(
    nn.Linear(n_feature, 1)).to(device)
loss_fn = nn.MSELoss()
optimizer = Adam(model1.parameters(), lr=0.01)
print(model1)

Sequential(
  (0): Linear(in_features=3, out_features=1, bias=True)
)


In [13]:
def train(dataloader, model, loss_fn, optimizer, device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    tr_loss = 0
    model.train()
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        loss = loss_fn(pred, y)
        tr_loss += loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    tr_loss /= num_batches
    return tr_loss.item()

In [14]:
def evaluate(x_val_tensor, y_val_tensor, model, loss_fn, device):
  model.eval()
  with torch.no_grad():
    x, y = x_val_tensor.to(device), y_val_tensor.to(device)
    pred = model(x)
    val_loss = loss_fn(pred, y).item()
  return val_loss, pred


In [15]:
epochs = 50
tr_loss_list, val_loss_list = [], []
for t in range(epochs):
  tr_loss = train(train_loader, model1, loss_fn, optimizer, device)
  val_loss, _ = evaluate(x_val_ts, y_val_ts, model1, loss_fn, device)
  tr_loss_list.append(tr_loss)
  val_loss_list.append(val_loss)
  print(f"Epoch {t+1} | Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:4f}")


Epoch 1 | Train Loss: 588.4670 | Val Loss: 509.721191
Epoch 2 | Train Loss: 580.9565 | Val Loss: 499.527191
Epoch 3 | Train Loss: 566.9361 | Val Loss: 489.510010
Epoch 4 | Train Loss: 554.6012 | Val Loss: 479.693695
Epoch 5 | Train Loss: 547.6516 | Val Loss: 470.084473
Epoch 6 | Train Loss: 541.5666 | Val Loss: 460.626312
Epoch 7 | Train Loss: 531.6205 | Val Loss: 451.385559
Epoch 8 | Train Loss: 524.0598 | Val Loss: 442.342712
Epoch 9 | Train Loss: 510.6743 | Val Loss: 433.406403
Epoch 10 | Train Loss: 503.6640 | Val Loss: 424.751312
Epoch 11 | Train Loss: 489.6408 | Val Loss: 416.231659
Epoch 12 | Train Loss: 483.6076 | Val Loss: 407.980133
Epoch 13 | Train Loss: 471.0994 | Val Loss: 399.846863
Epoch 14 | Train Loss: 471.7997 | Val Loss: 391.912872
Epoch 15 | Train Loss: 467.2661 | Val Loss: 384.089447
Epoch 16 | Train Loss: 454.1159 | Val Loss: 376.438995
Epoch 17 | Train Loss: 444.2871 | Val Loss: 368.968903
Epoch 18 | Train Loss: 438.4044 | Val Loss: 361.722565
Epoch 19 | Train Lo